# **TAHAP 1**

# Set Up dan Download Library

In [146]:
!pip install pymupdf pandas scikit-learn nltk

In [147]:
import os

os.makedirs("/content/data/pdf", exist_ok=True)
os.makedirs("/content/data/raw", exist_ok=True)
os.makedirs("/content/data/processed/clean_text", exist_ok=True)
os.makedirs("/content/data/eval", exist_ok=True)
os.makedirs("/content/data/logs", exist_ok=True)

print("SETUP DONE")

SETUP DONE


# PDF ke TXT

In [148]:
import fitz
import os

input_dir = "/content/data/pdf"
output_dir = "/content/data/raw"

count = 0
failed = []

for file in os.listdir(input_dir):

    if file.endswith(".pdf"):

        try:
            doc = fitz.open(os.path.join(input_dir, file))
            text = ""

            for page in doc:
                text += page.get_text()

            doc.close()

            out_path = os.path.join(output_dir, file.replace(".pdf", ".txt"))

            with open(out_path, "w", encoding="utf-8") as f:
                f.write(text)

            count += 1

        except:
            failed.append(file)

print("BERHASIL:", count)
print("GAGAL:", len(failed))

BERHASIL: 34
GAGAL: 0


# Cek hasil Raw

In [149]:
with open("/content/data/raw/case_001.txt", encoding="utf-8") as f:
    print(f.read()[:1000])

hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id
P U T U S A N
Nomor 747/Pid.B/2025/PN Mtr
DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA
Pengadilan  Negeri  Mataram yang  mengadili  perkara  pidana  dengan
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
berikut dalam perkara Terdakwa :
1. Nama lengkap 
: ABDULLAH
2. Tempat lahir 
: Mentigi
3. Umur/Tanggal lahir 
: 46 Tahun /31 Desember 1979
4. Jenis kelamin 
: Laki-laki
5. Kebangsaan 
: Indonesia
6. Tempat tinggal 
: Dusun Mentigi, Desa Malaka, Kecamatan 
Pemenang, Kabupaten Lombok Utara
7. Agama 
: Islam
8. Pekerjaan 
: Wiraswasta
Terdakwa Abdullah ditahan dalam rumah tahanan oleh : 
1. Penyidik, sejak tanggal 29 Juli 2025 sampai dengan tanggal 17 Agustus
2025. 
2. Penyidik, Perpanjangan Oleh Penuntut Umum sejak tanggal 18 Agustus 2025
sampai dengan tanggal 26 September 2025

# Cleaning Text

In [150]:
import re
import os

def clean_text(text):

    text = text.lower()

    # FIX PDF GARBAGE (INI WAJIB)
    text = re.sub(r'\ba\s*h\s*agung\s*repub.*?indonesia\b',
                  'mahkamah agung republik indonesia', text)

    text = re.sub(r'direktori putusan.*?go\.id', ' ', text)
    text = re.sub(r'putusan\.mahkamahagung.*?go\.id', ' ', text)

    # hapus header umum
    text = re.sub(r'hkama', ' ', text)

    # hapus karakter aneh
    text = re.sub(r'[^a-z0-9\s\.\,\;\:\-\(\)]', ' ', text)

    # normalisasi spasi
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


raw_dir = "/content/data/raw"
clean_dir = "/content/data/processed/clean_text"

os.makedirs(clean_dir, exist_ok=True)

for file in os.listdir(raw_dir):
    if file.endswith(".txt"):

        with open(os.path.join(raw_dir, file), "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        cleaned = clean_text(text)

        with open(os.path.join(clean_dir, file), "w", encoding="utf-8") as f:
            f.write(cleaned)

print("CLEANING FIXED DONE")

CLEANING FIXED DONE


# Hasil Cleaning

In [151]:
file_path = "/content/data/processed/clean_text/case_001.txt"

with open(file_path, encoding="utf-8") as f:
    print(f.read()[:1000])

a h agung repub a h agung republik indonesia mah agung republik indonesia blik indonesi direktori putusan ma h agung republik indonesia p u t u s a n nomor 747 pid.b 2025 pn mtr demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri mataram yang mengadili perkara pidana dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai berikut dalam perkara terdakwa : 1. nama lengkap : abdullah 2. tempat lahir : mentigi 3. umur tanggal lahir : 46 tahun 31 desember 1979 4. jenis kelamin : laki-laki 5. kebangsaan : indonesia 6. tempat tinggal : dusun mentigi, desa malaka, kecamatan pemenang, kabupaten lombok utara 7. agama : islam 8. pekerjaan : wiraswasta terdakwa abdullah ditahan dalam rumah tahanan oleh : 1. penyidik, sejak tanggal 29 juli 2025 sampai dengan tanggal 17 agustus 2025. 2. penyidik, perpanjangan oleh penuntut umum sejak tanggal 18 agustus 2025 sampai dengan tanggal 26 september 2025. 3. penuntut umum, sejak tanggal 26 september 2025 sampai denga

In [152]:
print("RAW FILES:", len(os.listdir("/content/data/raw")))
print("CLEAN FILES:", len(os.listdir("/content/data/processed/clean_text")))

RAW FILES: 34
CLEAN FILES: 34


# VALIDASI KEUTUHAN TEKS


In [153]:
valid = []
invalid = []

for file in os.listdir(clean_dir):

    if not file.endswith(".txt"):
        continue

    path = os.path.join(clean_dir, file)

    with open(path, encoding="utf-8") as f:
        text = f.read()

    if len(text.split()) > 500:
        valid.append(file)
    else:
        invalid.append(file)

print("VALID:", len(valid))
print("INVALID:", len(invalid))

VALID: 34
INVALID: 0


# **TAHAP 2**

In [154]:
import os
import re
import pandas as pd

# Load Data Clean

In [155]:
clean_dir = "/content/data/processed/clean_text"

# Normalisasi

In [156]:
def normalize(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text

# Case Representation

In [157]:
rows = []

for file in os.listdir(clean_dir):

    if not file.endswith(".txt"):
        continue

    with open(
        os.path.join(clean_dir, file),
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:
        text = f.read()

    text = normalize(text)

    # METADATA

    no_perkara = ""
    tanggal = ""
    terdakwa = ""

    m_no = re.search(
        r'putusan\s*nomor\s*[:\-]?\s*([0-9/\.a-z]+)',
        text
    )

    if m_no:
        no_perkara = m_no.group(1)

    m_tgl = re.search(
        r'tanggal\s*[:\-]?\s*([^,\.]+)',
        text
    )

    if m_tgl:
        tanggal = m_tgl.group(1)

    m_td = re.search(
        r'nama lengkap\s*[:\-]?\s*(.*?)(tempat lahir|umur|jenis kelamin|kebangsaan)',
        text
    )

    if m_td:
        terdakwa = m_td.group(1).strip()

    # KONTEN KUNCI

    ringkasan_fakta = " ".join(
        text.split()[:200]
    )

    m_pasal = re.search(
        r'pasal\s*([^\n,\.]+)',
        text
    )

    pasal = (
        m_pasal.group(1).strip()
        if m_pasal else ""
    )

    m_amar = re.search(
        r'amar\s*putusan(.*?)(menimbang|demikian|putusan|$)',
        text
    )

    amar_putusan = (
        m_amar.group(1).strip()[:800]
        if m_amar else ""
    )

    # argumen hukum utama

    argumen_hukum = (
        pasal + " " + amar_putusan
    ).strip()

    # FEATURE ENGINEERING

    word_count = len(text.split())
    char_count = len(text)

    # JENIS PERKARA

    jenis_perkara = "pidana umum"

    if "narkotika" in text:
        jenis_perkara = "pidana narkotika"

    # SIMPAN ROW

    rows.append({

        "case_id": file.replace(".txt", ""),
        "no_perkara": no_perkara,
        "tanggal": tanggal,
        "jenis_perkara": jenis_perkara,
        "terdakwa": terdakwa,
        "pasal": pasal,
        "ringkasan_fakta": ringkasan_fakta,
        "argumen_hukum": argumen_hukum,
        "amar_putusan": amar_putusan,
        "word_count": word_count,
        "char_count": char_count,
        "text_full": text
    })

In [158]:
df = pd.DataFrame(rows)

output_path = "/content/data/processed/cases.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("SAVED:", output_path)
print("SHAPE:", df.shape)

df.head()

SAVED: /content/data/processed/cases.csv
SHAPE: (34, 12)


,case_id,no_perkara,tanggal,jenis_perkara,terdakwa,pasal,ringkasan_fakta,argumen_hukum,amar_putusan,word_count,char_count,text_full
0,case_026,352,lahir : 45 tahun 8 agustus 1979; 4,pidana umum,fathurrahman; 2.,335 ayat (1) kuhp,a h agung repub a h agung republik indonesia m...,335 ayat (1) kuhp,,5457,37202,a h agung repub a h agung republik indonesia m...
1,case_015,450,lahir : 50 tahun 1 juli 1975 4,pidana umum,muhadi 2.,170 ayat (1) kuhp jo,a h agung repub a h agung republik indonesia m...,170 ayat (1) kuhp jo,,23271,162746,a h agung repub a h agung republik indonesia m...
2,case_023,319,30 maret 2018; terdakwa ditahan di rumah tahan...,pidana umum,murnawadi,335 ayat (1) ke -1 kuhp; 2,a h agung repub a h agung republik indonesia m...,335 ayat (1) ke -1 kuhp; 2 ini;,ini;,5543,38219,a h agung repub a h agung republik indonesia m...
3,case_009,319,30 maret 2018; terdakwa ditahan di rumah tahan...,pidana umum,murnawadi,335 ayat (1) ke -1 kuhp; 2,a h agung repub a h agung republik indonesia m...,335 ayat (1) ke -1 kuhp; 2 ini;,ini;,5543,38219,a h agung repub a h agung republik indonesia m...
4,case_001,747,lahir : 46 tahun 31 desember 1979 4,pidana umum,abdullah 2.,335 ayat (1) ke-1 kuhp,a h agung repub a h agung republik indonesia m...,335 ayat (1) ke-1 kuhp ini ;,ini ;,10273,71216,a h agung repub a h agung republik indonesia m...


In [160]:
print(df["jenis_perkara"].value_counts())

jenis_perkara
pidana umum    34
Name: count, dtype: int64


# **TAHAP 3**

In [161]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load Case Base

In [162]:
df = pd.read_csv("/content/data/processed/cases.csv")

df.head()

,case_id,no_perkara,tanggal,jenis_perkara,terdakwa,pasal,ringkasan_fakta,argumen_hukum,amar_putusan,word_count,char_count,text_full
0,case_026,352.0,lahir : 45 tahun 8 agustus 1979; 4,pidana umum,fathurrahman; 2.,335 ayat (1) kuhp,a h agung repub a h agung republik indonesia m...,335 ayat (1) kuhp,NaN,5457,37202,a h agung repub a h agung republik indonesia m...
1,case_015,450.0,lahir : 50 tahun 1 juli 1975 4,pidana umum,muhadi 2.,170 ayat (1) kuhp jo,a h agung repub a h agung republik indonesia m...,170 ayat (1) kuhp jo,NaN,23271,162746,a h agung repub a h agung republik indonesia m...
2,case_023,319.0,30 maret 2018; terdakwa ditahan di rumah tahan...,pidana umum,murnawadi,335 ayat (1) ke -1 kuhp; 2,a h agung repub a h agung republik indonesia m...,335 ayat (1) ke -1 kuhp; 2 ini;,ini;,5543,38219,a h agung repub a h agung republik indonesia m...
3,case_009,319.0,30 maret 2018; terdakwa ditahan di rumah tahan...,pidana umum,murnawadi,335 ayat (1) ke -1 kuhp; 2,a h agung repub a h agung republik indonesia m...,335 ayat (1) ke -1 kuhp; 2 ini;,ini;,5543,38219,a h agung repub a h agung republik indonesia m...
4,case_001,747.0,lahir : 46 tahun 31 desember 1979 4,pidana umum,abdullah 2.,335 ayat (1) ke-1 kuhp,a h agung repub a h agung republik indonesia m...,335 ayat (1) ke-1 kuhp ini ;,ini ;,10273,71216,a h agung repub a h agung republik indonesia m...


# Siapkan Data Teks

In [163]:
df["combined_text"] = (

    df["ringkasan_fakta"].fillna("").astype(str)
    + " " +

    df["argumen_hukum"].fillna("").astype(str)
    + " " +

    df["pasal"].fillna("").astype(str)
    + " " +

    df["amar_putusan"].fillna("").astype(str)
    + " " +

    df["text_full"].fillna("").astype(str)

)

In [165]:
from sklearn.model_selection import train_test_split

train_df,test_df=train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(train_df.shape)
print(test_df.shape)

print("TRAIN:", len(train_df))
print("TEST :", len(test_df))
print("RATIO:", round(len(train_df)/len(df),2))

(27, 13)
(7, 13)
TRAIN: 27
TEST : 7
RATIO: 0.79


# TF-IDF Vector

In [166]:
vectorizer = TfidfVectorizer(
    max_features=5000
)

train_matrix = vectorizer.fit_transform(
    train_df["combined_text"]
)

print("TFIDF:", train_matrix.shape)

TFIDF: (27, 5000)


# Function Retrieval

In [167]:
def retrieve_case(query, top_k=5):

    query = str(query).lower()

    query_vec = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vec,
        train_matrix
    ).flatten()

    top_idx = scores.argsort()[::-1][:top_k]

    results = train_df.iloc[top_idx].copy()

    results["similarity_score"] = scores[top_idx]

    return results[
        [
            "case_id",
            "no_perkara",
            "terdakwa",
            "pasal",
            "similarity_score"
        ]
    ]

# TEST QUERY

In [168]:
query = """
pencurian dengan pemberatan pasal 363 KUHP
terdakwa mengambil barang tanpa izin
"""

retrieve_case(query, top_k=5)

,case_id,no_perkara,terdakwa,pasal,similarity_score
13,case_033,NaN,mardip; 2.,335 ayat (1) kuhp sebagaimana dalam dakwaan ja...,0.096829
33,case_030,555.0,sahban 2.,335 ayat (1) ke-1 kuhp ; halaman 1 dari 25 put...,0.096619
7,case_010,631.0,NaN,368 ayat (1) kuhp; 2,0.094723
11,case_021,631.0,NaN,368 ayat (1) kuhp; 2,0.094723
20,case_013,410.0,wildan;,335 ayat (1) ke 1 kuhp sebagaimana dalam dakwa...,0.094314


In [169]:
import json
import os

test_queries = [

{
"id":1,
"query":"pencurian pasal 363",
"ground_truth":"pidana umum"
},

{
"id":2,
"query":"penganiayaan",
"ground_truth":"pidana umum"
},

{
"id":3,
"query":"penipuan",
"ground_truth":"pidana umum"
},

{
"id":4,
"query":"penggelapan",
"ground_truth":"pidana umum"
},

{
"id":5,
"query":"narkotika",
"ground_truth":"pidana narkotika"
}

]

with open(
"/content/data/eval/queries.json",
"w",
encoding="utf-8"
) as f:

    json.dump(
        test_queries,
        f,
        indent=4,
        ensure_ascii=False
    )

print("queries.json saved")

queries.json saved


# **TAHAP 4**

In [170]:
import pandas as pd
from collections import Counter

# Load Data dan buat case_solution

In [171]:
df = pd.read_csv("/content/data/processed/cases.csv")

df["amar_putusan"] = df["amar_putusan"].fillna("").astype(str)

case_solutions = dict(
    zip(
        train_df["case_id"],
        train_df["amar_putusan"]
    )
)


# Fungsi Reuse

In [172]:
def predict_outcome(query, k=5):

    results = retrieve_case(query, top_k=k)

    case_ids = results["case_id"].astype(str).values
    scores = results["similarity_score"].values

    weighted_solutions = {}

    for cid, score in zip(case_ids, scores):

        solution = case_solutions.get(cid, None)

        if solution is None:
            continue

        solution = str(solution).strip()

        if solution == "" or solution.lower() == "nan":
            continue

        weighted_solutions[solution] = weighted_solutions.get(solution, 0) + score

    if weighted_solutions:
        predicted_solution = max(weighted_solutions, key=weighted_solutions.get)
    else:
        predicted_solution = "No solution found"

    return {
        "predicted_solution": predicted_solution,
        "top_case_ids": list(case_ids)
    }

# Test Case

In [173]:
query = """
pencurian dengan pemberatan pasal 363 KUHP
mengambil barang tanpa izin pemilik
"""

result = predict_outcome(query, k=5)

print("TOP CASE IDS:")
print(result["top_case_ids"])

print("\nPREDICTED SOLUTION:\n")
print(result["predicted_solution"][:800])

TOP CASE IDS:
['case_007', 'case_010', 'case_021', 'case_034', 'case_033']

PREDICTED SOLUTION:

ini; memperhatikan pasal 368ayat (1) kuhp, undang-undang nomor08 tahun 1981 tentang hukum acara pidana serta peraturan perundang-undangan lain yang bersangkutan; m e n g a d i l i 1. menyatakan terdakwa saleh tersebut di atas terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana pemerasan ; 2. menjatuhkan pidana terhadapterdakwa saleh dengan pidana penjara selama 1 (satu) tahun; 3. menetapkan masa penangkapan dan penahanan yang telah dijalani terdakwadikurangkan seluruhnya dari pidana yang dijatuhkan; 4. menetapkan terdakwa tetap berada dalam tahanan; 5. menetapkan barang bukti berupa : - 1 (satu) bendel karcis jasa parkir truck forum bertais rembuk mataram yang belum terpakai; - 1 (satu) lembar karcis jasa parkir truck forum bertais rembuk mataram yang sudah terpakai no.0109


# Simpan Hasil

In [174]:
test_queries = [
"pencurian pasal 363",
"penganiayaan",
"penipuan",
"penggelapan",
"narkotika"
]

results = []

for i, q in enumerate(test_queries):

    pred = predict_outcome(q, k=5)

    results.append({
        "query_id": i + 1,
        "query": q,
        "predicted_solution": pred["predicted_solution"],
        "top_5_case_ids": ",".join(pred["top_case_ids"])
    })

# buat dataframe hasil
df_result = pd.DataFrame(results)

print(df_result)

   query_id                query        predicted_solution  \
0         1  pencurian pasal 363                      ini;   
1         2         penganiayaan              dibawah ini;   
2         3             penipuan                         ;   
3         4          penggelapan  ; halaman 14 dari 16 hal   
4         5            narkotika  ; halaman 14 dari 16 hal   

                                 top_5_case_ids  
0  case_034,case_016,case_027,case_005,case_025  
1  case_033,case_006,case_001,case_010,case_030  
2  case_031,case_007,case_034,case_025,case_017  
3  case_007,case_017,case_025,case_010,case_030  
4  case_017,case_025,case_007,case_010,case_030  


In [175]:
import os

os.makedirs(
    "/content/data/results",
    exist_ok=True
)

output_path = "/content/data/results/predictions.csv"

df_result.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSAVED:", output_path)


SAVED: /content/data/results/predictions.csv


# **TAHAP 5**

In [176]:
import os
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

os.makedirs(
    "/content/data/eval",
    exist_ok=True
)

# Buat Ground Truth Query

In [177]:
queries = [

{
"id":1,
"query":"pencurian pasal 363",
"expected":"pidana umum"
},

{
"id":2,
"query":"penganiayaan",
"expected":"pidana umum"
},

{
"id":3,
"query":"penipuan",
"expected":"pidana umum"
},

{
"id":4,
"query":"penggelapan",
"expected":"pidana umum"
},

{
"id":5,
"query":"pemerasan dan pengancaman",
"expected":"pidana umum"
}

]

eval_rows=[]

# Evaluasi Retrieval

In [178]:
for q in queries:

    top = retrieve_case(
        q["query"],
        top_k=1
    )

    retrieved_id = top.iloc[0]["case_id"]

    pred_label = train_df[
        train_df["case_id"]==retrieved_id
    ]["jenis_perkara"].values[0]

    eval_rows.append({

        "query_id":q["id"],

        "query":q["query"],

        "actual":q["expected"],

        "predicted":pred_label
    })

retrieval_eval = pd.DataFrame(
    eval_rows
)

retrieval_eval

,query_id,query,actual,predicted
0,1,pencurian pasal 363,pidana umum,pidana umum
1,2,penganiayaan,pidana umum,pidana umum
2,3,penipuan,pidana umum,pidana umum
3,4,penggelapan,pidana umum,pidana umum
4,5,pemerasan dan pengancaman,pidana umum,pidana umum


# Hitung Accuracy Precision Recall F1

In [179]:
acc = accuracy_score(
    retrieval_eval["actual"],
    retrieval_eval["predicted"]
)

prec = precision_score(
    retrieval_eval["actual"],
    retrieval_eval["predicted"],
    average="weighted",
    zero_division=0
)

rec = recall_score(
    retrieval_eval["actual"],
    retrieval_eval["predicted"],
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    retrieval_eval["actual"],
    retrieval_eval["predicted"],
    average="weighted",
    zero_division=0
)

metric_df = pd.DataFrame({

"accuracy":[acc],
"precision":[prec],
"recall":[rec],
"f1_score":[f1]

})

metric_df

,accuracy,precision,recall,f1_score
0,1.0,1.0,1.0,1.0


# Simpan Retrieval Metrics

In [180]:
metric_df.to_csv(
"/content/data/eval/retrieval_metrics.csv",
index=False
)

print(
"saved retrieval metrics"
)

saved retrieval metrics


# Evaluasi Prediction

In [181]:
prediction_rows=[]

for q in queries:

    pred = predict_outcome(
        q["query"]
    )

    label = "pidana umum"

    if "narkotika" in pred[
        "predicted_solution"
    ].lower():

        label="pidana narkotika"

    prediction_rows.append({

        "query_id":q["id"],

        "actual":q["expected"],

        "predicted":label
    })

prediction_eval=pd.DataFrame(
    prediction_rows
)

prediction_eval

,query_id,actual,predicted
0,1,pidana umum,pidana umum
1,2,pidana umum,pidana umum
2,3,pidana umum,pidana umum
3,4,pidana umum,pidana umum
4,5,pidana umum,pidana umum


# Hitung Metric Prediction

In [182]:
acc2 = accuracy_score(
prediction_eval["actual"],
prediction_eval["predicted"]
)

prec2 = precision_score(
prediction_eval["actual"],
prediction_eval["predicted"],
average="weighted",
zero_division=0
)

rec2 = recall_score(
prediction_eval["actual"],
prediction_eval["predicted"],
average="weighted",
zero_division=0
)

f12 = f1_score(
prediction_eval["actual"],
prediction_eval["predicted"],
average="weighted",
zero_division=0
)

prediction_metric=pd.DataFrame({

"accuracy":[acc2],
"precision":[prec2],
"recall":[rec2],
"f1_score":[f12]

})

prediction_metric

,accuracy,precision,recall,f1_score
0,1.0,1.0,1.0,1.0


# Simpan Prediction Metrics

In [183]:
prediction_metric.to_csv(
"/content/data/eval/prediction_metrics.csv",
index=False
)

print(
"SAVED prediction metrics"
)

SAVED prediction metrics


# SIMPAN DETAIL EVALUASI

In [184]:
retrieval_eval.to_csv(
    "/content/data/eval/retrieval_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

prediction_eval.to_csv(
    "/content/data/eval/prediction_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

print("DETAIL EVALUATION SAVED")


DETAIL EVALUATION SAVED


# Error Analysis

In [185]:
print("\n=== ERROR ANALYSIS ===")

failed = prediction_eval[
    prediction_eval["actual"]
    !=
    prediction_eval["predicted"]
]

if len(failed) == 0:

    print(
        "Tidak ada prediksi gagal"
    )

else:

    print(
        "Jumlah gagal:",
        len(failed)
    )

    print(
        failed[
            [
                "query_id",
                "actual",
                "predicted"
            ]
        ]
    )


=== ERROR ANALYSIS ===
Tidak ada prediksi gagal


In [186]:
print(df["jenis_perkara"].value_counts())

jenis_perkara
pidana umum    34
Name: count, dtype: int64
